In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns',None)

#load processed data
train = pd.read_csv('../data/train_processed.csv')
test  = pd.read_csv('../data/processed_test.csv')
y_train = np.load('../data/y_train.npy')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Target shape:', y_train.shape)

Train shape: (1458, 172)
Test shape: (1459, 172)
Target shape: (1458,)


In [2]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10))
])

#cross validate
scores = cross_val_score(
    baseline,
    train,
    y_train,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

rmsle = -scores.mean()
std   = scores.std()

print(f'Ridge Baseline RMSLE: {rmsle:.4f} ± {std:.4f}')

Ridge Baseline RMSLE: 0.1212 ± 0.0061


In [3]:
from xgboost import XGBRegressor

#XGBoost baseline
xgb_model = XGBRegressor(
    n_estimators = 1000,
    learning_rate = 0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(
    xgb_model,
    train,
    y_train,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

rmsle = -scores.mean()
std   = scores.std()

print(f'XGBoost RMSLE: {rmsle:.4f} ± {std:.4f}')
print(f'Improvement over Ridge: {0.1212 - rmsle:.4f}')

XGBoost RMSLE: 0.1287 ± 0.0084
Improvement over Ridge: -0.0075


In [4]:
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
import numpy as np

# use early stopping — stops adding trees when
# validation score stops improving
# this prevents overfitting automatically!

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores=[]

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    X_fold_train = train.iloc[train_idx]
    X_fold_val   = train.iloc[val_idx]
    y_fold_train = y_train[train_idx]
    y_fold_val   = y_train[val_idx]

    xgb = XGBRegressor(
        n_estimators=3000,        # high ceiling
        learning_rate=0.05,
        max_depth=4,              # shallower trees
        subsample=0.8,            # use 80% of rows per tree
        colsample_bytree=0.8,     # use 80% of features per tree
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50  # stop if no improvement in 50 rounds
    )

    xgb.fit(
        X_fold_train, y_fold_train,
        eval_set=[(X_fold_val, y_fold_val)],
        verbose=False
    )
    
    preds = xgb.predict(X_fold_val)
    fold_score = np.sqrt(mean_squared_error(y_fold_val, preds))
    scores.append(fold_score)
    print(f'Fold {fold+1}: {fold_score:.4f} — best iteration: {xgb.best_iteration}')

print(f'\nXGBoost with early stopping: {np.mean(scores):.4f} ± {np.std(scores):.4f}')

Fold 1: 0.1230 — best iteration: 163
Fold 2: 0.1101 — best iteration: 260
Fold 3: 0.1199 — best iteration: 498
Fold 4: 0.1269 — best iteration: 206
Fold 5: 0.1084 — best iteration: 213

XGBoost with early stopping: 0.1176 ± 0.0073


In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # define search space — ranges for each parameter
    params = {
        'n_estimators'      : 3000,
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 8),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0, 3),
        'random_state'      : 42,
        'n_jobs'            : -1,
        'early_stopping_rounds': 50
    }
    
    # cross validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(train):
        X_fold_train = train.iloc[train_idx]
        X_fold_val   = train.iloc[val_idx]
        y_fold_train = y_train[train_idx]
        y_fold_val   = y_train[val_idx]
        
        model = XGBRegressor(**params)
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=[(X_fold_val, y_fold_val)],
            verbose=False
        )
        
        preds = model.predict(X_fold_val)
        score = np.sqrt(mean_squared_error(y_fold_val, preds))
        scores.append(score)
    
    return np.mean(scores)

# run optimization — 50 trials
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest RMSLE: {study.best_value:.4f}')
print(f'Best parameters:')
for key, val in study.best_params.items():
    print(f'  {key}: {val}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best RMSLE: 0.1142
Best parameters:
  learning_rate: 0.04887476037041333
  max_depth: 3
  subsample: 0.6198606629858174
  colsample_bytree: 0.6769480142016289
  min_child_weight: 10
  reg_alpha: 0.44170294579462527
  reg_lambda: 0.12091936165446476


In [6]:
# train final model on ALL training data
# using best parameters from Optuna
best_params = study.best_params
best_params.update({
    'n_estimators'        : 3000,
    'random_state'        : 42,
    'n_jobs'              : -1,
    'early_stopping_rounds': 50
})

# we need a validation set for early stopping
# use 10% of training data
from sklearn.model_selection import train_test_split

X_train_final, X_val_final, y_train_final, y_val_final = \
    train_test_split(train, y_train, 
                     test_size=0.1, 
                     random_state=42)

# train final model
final_model = XGBRegressor(**best_params)
final_model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_val_final, y_val_final)],
    verbose=100  # print every 100 trees
)

print(f'\nBest iteration: {final_model.best_iteration}')

[0]	validation_0-rmse:0.40197
[100]	validation_0-rmse:0.12109
[200]	validation_0-rmse:0.11166
[300]	validation_0-rmse:0.10898
[349]	validation_0-rmse:0.10917

Best iteration: 299


In [ ]:
#make predictions on test data
test_predictions_log = final_model.predict(test)

#reverse log transformation → back to actual prices
test_predictions = np.expm1(test_predictions_log)

print(f'Prediction stats:')
print(f'Min:  ${test_predictions.min():,.0f}')
print(f'Max:  ${test_predictions.max():,.0f}')
print(f'Mean: ${test_predictions.mean():,.0f}')

Prediction stats:
Min:  $49,070
Max:  $540,209
Mean: $178,378


In [8]:
#load original test data to get Id column
test_original = pd.read_csv('../data/test.csv')

#create submission dataframe
submission = pd.DataFrame({
    'Id'       : test_original['Id'],
    'SalePrice': test_predictions
})

#save
submission.to_csv('../data/submission.csv', index=False)

print('Submission file created!')
print(f'Shape: {submission.shape}')
print(submission.head(10))

Submission file created!
Shape: (1459, 2)
     Id      SalePrice
0  1461  120778.078125
1  1462  161925.500000
2  1463  184848.343750
3  1464  191239.359375
4  1465  181919.015625
5  1466  171931.015625
6  1467  178156.562500
7  1468  170871.125000
8  1469  184594.671875
9  1470  128007.195312
